# Bài 3 — Đóng gói và CI

**Tuần 22** · Chuẩn bị cho mini-project và đưa **P6** lên internet

Bài này khác hai bài trước: phần lớn nội dung là **file cấu hình**, không phải
code Python. Nhưng những file đó quyết định liệu API key của bạn có nằm vĩnh viễn
trong một ảnh Docker công khai hay không.

> 📌 Notebook không chạy được `docker build` (cần Docker daemon). Thay vào đó nó
> **kiểm tra các file mẫu** và chạy đúng những phép kiểm mà CI của bạn sẽ chạy.

In [ ]:
import re
import subprocess
import sys
from pathlib import Path

GOC = Path.cwd()
while not (GOC / "pyproject.toml").exists() and GOC != GOC.parent:
    GOC = GOC.parent
MAU = GOC / "curriculum" / "10-deploy-ops" / "mau"

for _l in (sys.stdout, sys.stderr):
    if (getattr(_l, "encoding", "") or "").lower().replace("-", "") != "utf8":
        try:
            _l.reconfigure(encoding="utf-8", errors="replace")
        except Exception:
            pass

print(f"Thư mục mẫu: {MAU}")
for f in sorted(MAU.iterdir()):
    print(f"   {f.name:<20} {f.stat().st_size:>6,} byte")

## 1. Vì sao Docker?

Câu trả lời thật lòng: **để "chạy được trên máy tôi" không còn là một câu nói đùa.**

| Không có Docker | Có Docker |
|---|---|
| "Máy tôi Python 3.11, máy chủ 3.9" | Cùng một Python |
| "Thiếu thư viện hệ thống nào đó" | Đã nằm trong ảnh |
| Cài đặt thủ công mỗi lần deploy | `docker run` |
| Rollback = cài lại từ đầu | Rollback = chạy ảnh cũ |

Dòng cuối là dòng đáng giá nhất khi có sự cố lúc 2 giờ sáng.

## 2. Build nhiều tầng

In [ ]:
noi_dung = (MAU / "Dockerfile").read_text(encoding="utf-8")
print(noi_dung)

Ba quyết định trong file trên, mỗi cái giải quyết một vấn đề cụ thể:

### ① Hai tầng

Tầng build có compiler và header để cài gói; tầng chạy thì không. Ảnh cuối nhỏ hơn
đáng kể và **ít thứ để tấn công hơn** — một compiler trong ảnh production là một
công cụ sẵn có cho kẻ xâm nhập.

### ② `COPY pyproject.toml` trước, `COPY . .` sau

Docker cache theo từng lệnh. Copy cả thư mục trước là **tự vứt bỏ cache mỗi lần
sửa một dòng code** — và bước `pip install` nặng nề phải chạy lại từ đầu.

### ③ `USER ungdung`

Mặc định container chạy bằng **root**. Nếu ai đó thoát được ra khỏi ứng dụng, họ
sẽ là root của container thay vì một người dùng không có quyền gì.

In [ ]:
# Kiểm tra Dockerfile có đủ các thực hành tốt không.
# CHỈ xét các dòng LỆNH - bỏ chú thích, nếu không thì chính phần cảnh báo
# "đừng làm thế này" trong file sẽ bị đếm là vi phạm.
cac_lenh = [d for d in noi_dung.splitlines()
            if d.strip() and not d.strip().startswith("#")]
than = chr(10).join(cac_lenh)
# HEALTHCHECK trai dai nhieu dong (co dau \ noi tiep) - phai gop lai,
# neu khong thi ban chi doc duoc dong dau va bo mat phan CMD.
i_hc = next((i for i, d in enumerate(cac_lenh)
             if d.strip().startswith("HEALTHCHECK")), None)
khoi_healthcheck = " ".join(cac_lenh[i_hc:i_hc + 3]) if i_hc is not None else ""

kiem = {
    "Build nhiều tầng": "AS builder" in than,
    "Không chạy bằng root": "USER " in than,
    "Copy khai báo phụ thuộc trước":
        than.index("COPY pyproject.toml") < than.index("COPY --chown"),
    "Có HEALTHCHECK": "HEALTHCHECK" in than,
    "Healthcheck dùng /khoe, không phải /san_sang":
        "/khoe" in khoi_healthcheck and "/san_sang" not in khoi_healthcheck,
    "KHÔNG có API key trong ENV/ARG":
        not re.search(r"(ENV|ARG)\s+\w*API_KEY\s*=", than),
}
for ten, dat in kiem.items():
    print(f'{"✅" if dat else "❌"} {ten}')

> ⚠️ **Dòng "Healthcheck dùng /khoe"** là chỗ rất dễ làm sai.
>
> `HEALTHCHECK` thất bại nghĩa là **khởi động lại container**. Nếu bạn trỏ nó vào
> `/san_sang`, thì mỗi lần Anthropic gián đoạn, toàn bộ instance của bạn bị khởi
> động lại đồng loạt — mất cache, mất connection pool, và khi dịch vụ phụ thuộc
> hồi phục thì bạn vẫn đang khởi động.
>
> Một sự cố 5 phút của người khác vừa thành sự cố 20 phút của bạn.

## 3. `.dockerignore` — file quan trọng nhất mà ai cũng quên

In [ ]:
bo_qua = (MAU / ".dockerignore").read_text(encoding="utf-8")
print(bo_qua)

> ⚠️ **Đây không phải chuyện kích thước ảnh.**
>
> Ảnh Docker là một chuỗi **tầng**, và mỗi tầng đọc lại được. Xoá file ở tầng sau
> **không** xoá nó khỏi tầng trước — `docker history --no-trunc` vẫn lấy ra được.
>
> Nghĩa là: một file `.env` lọt vào build context sẽ nằm trong ảnh **vĩnh viễn**,
> kể cả khi dòng lệnh tiếp theo là `RUN rm .env`.

Ba mục nguy hiểm nhất trong danh sách:

| Mục | Vì sao |
|---|---|
| `.env` | Chứa key. Hiển nhiên, và vẫn bị quên |
| `.git` | Chứa **toàn bộ lịch sử** — kể cả key đã bị xoá ở commit sau |
| `data/` | Có thể chứa dữ liệu người dùng, và làm ảnh nặng gấp nhiều lần |

Mục `.git` là mục tinh vi nhất. Bạn commit nhầm key, phát hiện ra, xoá đi và
commit lại — nhưng key vẫn nằm trong lịch sử Git, và Git đi vào ảnh Docker.

In [ ]:
# Chạy đúng phép kiểm mà CI sẽ chạy: tìm key thật bị commit nhầm
MAU_KEY = re.compile(r"sk-ant-api[0-9]{2}-[A-Za-z0-9_-]{20,}")

# Danh sách cho phép: các chuỗi CỐ Ý dùng làm ví dụ trong khoá học.
# Mọi bộ dò bí mật thật (gitleaks, trufflehog...) đều có cơ chế này - không có
# nó thì cảnh báo giả nhiều tới mức người ta tắt luôn công cụ.
CHO_PHEP = re.compile(r"(?i)(KHOA-GIA|vidu|example|placeholder|xxx|KHONG-THAT)")

nghi_ngo, bo_qua_vi_allowlist = [], 0
for f in GOC.rglob("*"):
    if not f.is_file() or f.suffix not in (".py", ".md", ".toml", ".yml", ".json"):
        continue
    if any(p in f.parts for p in (".git", "node_modules", ".venv", "venv")):
        continue
    try:
        for so_dong, dong in enumerate(f.read_text(encoding="utf-8").splitlines(), 1):
            if not MAU_KEY.search(dong):
                continue
            if CHO_PHEP.search(dong):
                bo_qua_vi_allowlist += 1
            else:
                nghi_ngo.append((f.relative_to(GOC), so_dong, dong.strip()[:60]))
    except (UnicodeDecodeError, PermissionError):
        pass

print(f"Bỏ qua vì nằm trong danh sách cho phép: {bo_qua_vi_allowlist} dòng")
if nghi_ngo:
    print(f"❌ Còn {len(nghi_ngo)} chuỗi cần xem lại:")
    for duong_dan, so_dong, trich in nghi_ngo[:10]:
        print(f"   {duong_dan}:{so_dong}  {trich}")
else:
    print("✅ Không còn chuỗi nào giống API key thật")

> 📌 **Con số "bỏ qua vì danh sách cho phép" chính là bài học ở đây.**
>
> Bộ dò bí mật nào cũng bắt nhầm — khoá học này cố tình dùng các chuỗi *trông
> giống* key thật để minh hoạ, và bộ dò không phân biệt được.
>
> Nếu bạn không có cơ chế cho phép, CI sẽ đỏ ở mọi lần push vì những cảnh báo giả.
> Và điều xảy ra tiếp theo luôn giống nhau: **người ta tắt luôn công cụ** — rồi
> lần key thật bị commit, không còn ai canh nữa.
>
> Mọi bộ dò thật (gitleaks, trufflehog, GitHub secret scanning) đều có allowlist
> vì lý do này. Một cảnh báo bị bỏ qua tệ hơn không có cảnh báo, vì nó cho bạn
> cảm giác an toàn giả.

Phép kiểm này chạy trong CI ở mọi lần push. Nó không hoàn hảo — nó chỉ bắt
được **định dạng key của Anthropic** — nhưng nó bắt được đúng lỗi phổ biến nhất:
dán key vào một notebook để "thử nhanh" rồi quên xoá.

> 📌 Nếu bạn **đã** commit một key thật: xoá khỏi code là chưa đủ. Phải **xoay
> key** (tạo key mới, thu hồi key cũ) ở Console. Key nằm trong lịch sử Git, trong
> bản sao của người khác, và có thể đã bị các bot quét GitHub thu thập.
>
> Xoay key mất 2 phút. Không xoay có thể mất cả hạn mức chi tiêu của bạn.

## 4. Bí mật: truyền lúc chạy, không phải lúc build

In [ ]:
VI_DU_SAI = [
    ("ENV ANTHROPIC_API_KEY=sk-ant-...", "Key nằm trong ảnh, vĩnh viễn"),
    ("COPY .env .", "Key nằm trong ảnh, vĩnh viễn"),
    ("ARG API_KEY", "Key nằm trong lịch sử build, đọc được bằng docker history"),
    ("docker build --build-arg KEY=...", "Như trên"),
]

print("❌ KHÔNG BAO GIỜ:")
for lenh, vi_sao in VI_DU_SAI:
    print(f"   {lenh:<38} {vi_sao}")

print()
print("✅ Truyền lúc CHẠY:")
for lenh in ["docker run --env-file .env dich-vu-llm",
             "docker run -e ANTHROPIC_API_KEY=$KEY dich-vu-llm",
             "(hoặc) secret manager của nền tảng: Fly, Railway, ECS, Kubernetes"]:
    print(f"   {lenh}")

## 5. CI: và vì sao nó KHÔNG được gọi API thật

In [ ]:
ci = (MAU / "ci.yml").read_text(encoding="utf-8")
print(ci[:2200])
print("...")

> ⭐ **Nguyên tắc xuyên suốt: CI không được gọi API thật.**
>
> | Lý do | Hậu quả nếu vi phạm |
> |---|---|
> | Chậm và không ổn định | Test đỏ vì mạng, không phải vì code |
> | Tốn tiền mỗi lần push | Hoá đơn tăng theo số lần bạn sửa typo |
> | Cần API key trong CI | Thêm một chỗ để lộ key |
>
> Đó chính là lý do **mọi bài tập từ Phase 07 đến giờ đều nhận hàm gọi model làm
> tham số**. Không phải để cho đẹp — mà để bộ test của bạn chạy được ở nơi không
> có mạng và không có tiền.

Chú ý biến `ANTHROPIC_API_KEY` trong CI là một **key giả**. Nếu một test nào đó
lỡ gọi API thật, nó sẽ thất bại ngay — và đó chính là điều bạn muốn: **CI phát
hiện ra, không phải hoá đơn phát hiện ra.**

In [ ]:
# Bộ test của khoá học này có gọi API thật không? Kiểm luôn.
so_test = 0
co_goi_that = []
for f in (GOC / "tests").rglob("test_*.py"):
    so_test += 1
    noi = f.read_text(encoding="utf-8")
    # Dấu hiệu gọi thật: tạo client Anthropic không qua tham số
    if "anthropic.Anthropic()" in noi or "messages.create(" in noi:
        co_goi_that.append(f.relative_to(GOC))

print(f"Đã quét {so_test} file test")
if co_goi_that:
    print(f"⚠️  {len(co_goi_that)} file có dấu hiệu gọi API thật:")
    for f in co_goi_that:
        print(f"   {f}")
else:
    print("✅ Không file test nào gọi API thật - CI chạy được miễn phí")

## 6. Danh sách kiểm trước khi lên production

Đây là danh sách bạn nên dán ở đâu đó và đi qua **từng dòng** trước mỗi lần deploy
lần đầu.

In [ ]:
DANH_SACH = [
    ("Bí mật", [
        "API key KHÔNG nằm trong code, ảnh Docker, hay lịch sử Git",
        ".env nằm trong .gitignore VÀ .dockerignore",
        "Log đã lọc secret (kiểm bằng cách đọc log thật, không phải đọc code)",
        "Đã đặt hạn mức chi tiêu ở Console Anthropic",
    ]),
    ("Cấu hình", [
        "Fail fast: cấu hình sai thì không khởi động",
        "GHI_LOG_PROMPT tắt ở prod",
        "Model ID không có hậu tố ngày tháng",
    ]),
    ("Khả dụng", [
        "/khoe và /san_sang là hai endpoint KHÁC nhau",
        "HEALTHCHECK trỏ vào /khoe",
        "Có timeout cho mọi lời gọi ra ngoài",
        "Có ngắt mạch cho dịch vụ phụ thuộc",
    ]),
    ("Chi phí", [
        "Có giới hạn độ dài đầu vào",
        "Có giới hạn tần suất",
        "Có ngân sách và nó CHẶN, không chỉ cảnh báo",
        "Đã ngoại suy chi phí cho lưu lượng dự kiến",
    ]),
    ("Quan sát", [
        "Log có cấu trúc, có trace_id",
        "Đo p95, không chỉ trung bình",
        "Có cảnh báo khi tỷ lệ lỗi hoặc chi phí vượt ngưỡng",
    ]),
    ("Vận hành", [
        "CI chạy test mà không gọi API thật",
        "Biết cách rollback về phiên bản trước",
        "Đã thử tắt dịch vụ phụ thuộc và xem ứng dụng phản ứng thế nào",
    ]),
]

tong = 0
for nhom, cac_muc in DANH_SACH:
    print(f"\n{nhom}")
    for m in cac_muc:
        tong += 1
        print(f"   [ ] {m}")
print(f"\n{tong} mục.")

> ⭐ **Mục cuối cùng là mục ít người làm nhất và đáng giá nhất:**
> *"đã thử tắt dịch vụ phụ thuộc và xem ứng dụng phản ứng thế nào"*.
>
> Bạn viết ngắt mạch, viết timeout, viết xử lý lỗi — nhưng cho tới khi thử tắt
> thật, bạn chỉ đang **hy vọng** chúng hoạt động.
>
> Cách thử rẻ nhất: đổi `ANTHROPIC_API_KEY` thành một chuỗi rác, hoặc trỏ
> `base_url` vào một cổng không có gì. Rồi nhìn xem: dịch vụ trả về gì? Sau bao
> lâu? Log có đủ để gỡ lỗi không? `/san_sang` có báo đúng không?

## Tóm tắt

| Điều | Nhớ gì |
|---|---|
| **Build nhiều tầng** | Ảnh nhỏ hơn và ít thứ để tấn công hơn |
| **Copy khai báo phụ thuộc trước** | Giữ cache Docker khi chỉ sửa code |
| **`USER` không phải root** | Giảm thiệt hại nếu ai đó thoát ra được |
| **`.dockerignore`** | Ảnh Docker đọc lại được từng tầng — `rm` không cứu được bạn |
| **Bí mật lúc chạy** | Không bao giờ `ENV`/`ARG`/`COPY .env` |
| **Đã lộ key thì XOAY key** | Xoá khỏi code là chưa đủ |
| **HEALTHCHECK → `/khoe`** | Trỏ vào `/san_sang` là tự tạo bão khởi động lại |
| **CI không gọi API thật** | Lý do mọi bài tập đều nhận hàm model làm tham số |

---

## Xong Phase 10

**Tiếp theo:**

- **Mini-project:** `curriculum/10-deploy-ops/mini_project/README.md`
- Đưa **P6** hoặc **P7** lên internet thật

> Bài test khắt khe nhất cho những gì bạn vừa học:
>
> Deploy xong, **đổi API key thành chuỗi rác** rồi gọi dịch vụ.
>
> Nếu nó trả về 503 kèm mã lỗi rõ ràng trong dưới một giây, `/san_sang` báo đỏ,
> `/khoe` vẫn xanh, và log có đủ thông tin để bạn biết chuyện gì xảy ra — bạn đã
> làm đúng.
>
> Nếu nó treo 30 giây rồi trả về một trang traceback, bạn còn việc phải làm.